# 지방간 위험도 예측 모델 학습

NHANES 데이터를 기반으로 지방간 단계(0~3)를 분류하는 모델을 학습합니다.

학습된 모델은 `ai_worker/models/` 에 저장되어 백엔드 API와 연결됩니다.

## 1. 환경 설정

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score
)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

DATA_PATH = '../data/nhanes_fatty_liver.csv'
MODEL_DIR = '../ai_worker/models'
os.makedirs(MODEL_DIR, exist_ok=True)

print('설정 완료')

## 2. 데이터 로드 및 탐색

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f'데이터 shape: {df.shape}')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# 결측치 확인
print('결측치:')
print(df.isnull().sum())

In [ ]:
# 타겟 분포
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['지방간단계명'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('지방간 단계 분포')
axes[0].set_xlabel('단계')
axes[0].set_ylabel('건수')
axes[0].tick_params(axis='x', rotation=0)

df['지방간단계명'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%')
axes[1].set_title('지방간 단계 비율')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
# 수치형 변수 분포
num_cols = ['나이', '키', '몸무게', 'BMI', '허리둘레', '1회음주량', '주당음주빈도', '월폭음빈도', '주당운동횟수', '평균수면시간']

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for i, col in enumerate(num_cols):
    ax = axes[i // 5][i % 5]
    df[col].hist(ax=ax, bins=30, color='steelblue', edgecolor='white')
    ax.set_title(col)

plt.tight_layout()
plt.show()

In [ ]:
# 상관관계 히트맵
plt.figure(figsize=(10, 8))
corr = df[num_cols + ['지방간단계']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('수치형 변수 상관관계')
plt.tight_layout()
plt.show()

## 3. 전처리

In [ ]:
# 특징 / 타겟 분리
TARGET = '지방간단계'
DROP_COLS = ['지방간단계', '지방간단계명']

X = df.drop(columns=DROP_COLS)
y = df[TARGET]

print(f'X shape: {X.shape}')
print(f'y 분포:\n{y.value_counts().sort_index()}')

In [ ]:
# 컬럼 타입 분류
NUMERIC_FEATURES = ['나이', '키', '몸무게', 'BMI', '허리둘레', '1회음주량', '주당음주빈도', '월폭음빈도', '주당운동횟수', '평균수면시간']

BINARY_FEATURES = ['성별', '음주여부', '운동여부', '흡연여부', '현재흡연여부', '당뇨진단여부', '고혈압진단여부', '수면장애여부']

ORDINAL_FEATURES = ['식습관자가평가']  # 좋음 > 보통 > 나쁨

print('수치형:', NUMERIC_FEATURES)
print('이진형:', BINARY_FEATURES)
print('순서형:', ORDINAL_FEATURES)

In [ ]:
# 범주형 변수 고유값 확인
for col in BINARY_FEATURES + ORDINAL_FEATURES:
    print(f'{col}: {df[col].unique().tolist()}')

In [ ]:
# Preprocessing Pipeline 정의
from sklearn.preprocessing import OrdinalEncoder

binary_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

# 식습관자가평가: 나쁨=0, 보통=1, 좋음=2
ordinal_encoder = OrdinalEncoder(
    categories=[['나쁨', '보통', '좋음']],
    handle_unknown='use_encoded_value',
    unknown_value=-1
)

numeric_scaler = StandardScaler()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_scaler, NUMERIC_FEATURES),
        ('bin', binary_encoder, BINARY_FEATURES),
        ('ord', ordinal_encoder, ORDINAL_FEATURES),
    ],
    remainder='drop'
)

print('Preprocessor 정의 완료')

In [ ]:
# Train / Test 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Train 타겟 분포:\n{y_train.value_counts().sort_index()}')

## 4. 모델 학습 및 비교

In [ ]:
models = {
    'RandomForest': RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=200, random_state=42, eval_metric='mlogloss', verbosity=0),
    'LightGBM': LGBMClassifier(n_estimators=200, random_state=42, verbosity=-1),
}

results = {}

for name, model in models.items():
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    
    # Cross-validation
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='f1_weighted', n_jobs=-1)
    
    # Test set 평가
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    test_acc = accuracy_score(y_test, y_pred)
    test_f1 = f1_score(y_test, y_pred, average='weighted')
    
    results[name] = {
        'pipeline': pipeline,
        'cv_f1_mean': cv_scores.mean(),
        'cv_f1_std': cv_scores.std(),
        'test_acc': test_acc,
        'test_f1': test_f1,
    }
    
    print(f'{name}: CV F1={cv_scores.mean():.4f}±{cv_scores.std():.4f} | Test Acc={test_acc:.4f} | Test F1={test_f1:.4f}')

In [ ]:
# 결과 비교 시각화
result_df = pd.DataFrame({
    name: {
        'CV F1 (mean)': v['cv_f1_mean'],
        'Test Accuracy': v['test_acc'],
        'Test F1': v['test_f1'],
    }
    for name, v in results.items()
}).T

result_df.plot(kind='bar', figsize=(10, 5), ylim=(0.5, 1.0))
plt.title('모델 성능 비교')
plt.xticks(rotation=0)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

result_df

In [ ]:
# 최고 성능 모델 선택
best_name = max(results, key=lambda k: results[k]['test_f1'])
best_pipeline = results[best_name]['pipeline']

print(f'최고 성능 모델: {best_name}')
print(f"Test F1: {results[best_name]['test_f1']:.4f}")

## 5. 최종 모델 상세 평가

In [ ]:
y_pred = best_pipeline.predict(X_test)
label_names = ['정상', '경도', '중등도', '중증']

print(classification_report(y_test, y_pred, target_names=label_names))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_names, yticklabels=label_names)
plt.title(f'{best_name} Confusion Matrix')
plt.xlabel('예측')
plt.ylabel('실제')
plt.tight_layout()
plt.show()

In [ ]:
# 특징 중요도 (트리 기반 모델)
feature_names = NUMERIC_FEATURES + BINARY_FEATURES + ORDINAL_FEATURES
classifier = best_pipeline.named_steps['classifier']

if hasattr(classifier, 'feature_importances_'):
    importances = classifier.feature_importances_
    fi_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
    fi_df = fi_df.sort_values('importance', ascending=True)

    plt.figure(figsize=(8, 7))
    plt.barh(fi_df['feature'], fi_df['importance'], color='steelblue')
    plt.title('특징 중요도')
    plt.xlabel('Importance')
    plt.tight_layout()
    plt.show()

## 6. 모델 저장

In [ ]:
import json

MODEL_PATH = os.path.join(MODEL_DIR, 'fatty_liver_model.pkl')
META_PATH = os.path.join(MODEL_DIR, 'model_meta.json')

# 모델 저장
joblib.dump(best_pipeline, MODEL_PATH)

# 메타 정보 저장 (API에서 참조)
meta = {
    'model_name': best_name,
    'test_accuracy': round(results[best_name]['test_acc'], 4),
    'test_f1': round(results[best_name]['test_f1'], 4),
    'label_map': {0: '정상', 1: '경도', 2: '중등도', 3: '중증'},
    'feature_order': feature_names,
    'numeric_features': NUMERIC_FEATURES,
    'binary_features': BINARY_FEATURES,
    'ordinal_features': ORDINAL_FEATURES,
}

with open(META_PATH, 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print(f'모델 저장: {MODEL_PATH}')
print(f'메타 저장: {META_PATH}')

## 7. 예측 함수 테스트

백엔드 AI Worker가 사용할 예측 함수와 동일한 인터페이스입니다.

In [ ]:
def predict_fatty_liver(input_data: dict) -> dict:
    """
    백엔드 API 입력값을 받아 지방간 위험도를 예측합니다.
    
    ai_worker/tasks/predict.py 와 동일한 인터페이스
    """
    model = joblib.load(MODEL_PATH)
    
    input_df = pd.DataFrame([input_data])
    
    stage = int(model.predict(input_df)[0])
    proba = model.predict_proba(input_df)[0].tolist()
    
    label_map = {0: '정상', 1: '경도', 2: '중등도', 3: '중증'}
    
    return {
        'stage': stage,
        'stage_label': label_map[stage],
        'probability': {
            label_map[i]: round(p, 4) for i, p in enumerate(proba)
        }
    }


# 테스트 입력
test_input = {
    '나이': 45.0,
    '성별': '남성',
    '키': 172.0,
    '몸무게': 85.0,
    'BMI': 28.7,
    '허리둘레': 92.0,
    '음주여부': '음주함',
    '1회음주량': 3.0,
    '주당음주빈도': 2.0,
    '월폭음빈도': 1.0,
    '운동여부': '운동안함',
    '주당운동횟수': 0,
    '흡연여부': '흡연경험있음',
    '현재흡연여부': '안함',
    '당뇨진단여부': '없음',
    '고혈압진단여부': '있음',
    '수면장애여부': '없음',
    '평균수면시간': 6.0,
    '식습관자가평가': '보통',
}

result = predict_fatty_liver(test_input)
print('예측 결과:')
print(f"  단계: {result['stage']} ({result['stage_label']})")
print(f"  확률: {result['probability']}")